# Palm Oil Grievance Classification: Fine-tuning TwHIN-BERT

## Overview

This notebook fine-tunes a pre-trained Twitter language model (**TwHIN-BERT**, `Twitter/twhin-bert-base`) to classify short palm-oil grievance texts into one of **6 topics** previously identified by BERTopic:

| Label | Topic |
|-------|-------|
| 0 | Failed Compensation / Land Rights |
| 1 | Environmental Impact |
| 2 | Administrative |
| 3 | Deforestation |
| 4 | Labour Rights |
| 5 | Illegal or Contaminated FFB |

## Data
* **309 labeled grievances**, split 80/20 into train (n=309 used here) and validation (n=78).
* Labels come from BERTopic dominant-topic assignments.
* Because the dataset is small and imbalanced, we (a) use **macro-F1** as the model-selection metric, (b) use **early stopping** on validation loss, and (c) use **Optuna** to search a small but reasonable hyper-parameter space.

## Pipeline
1. Install / import dependencies.
2. Load the original 80/20 train/validation CSVs.
3. **Audit the split** (deduplicate, check leakage, per-class distribution) and **re-split with stratification** so every class is represented near the target 20% in validation.
4. Tokenize with the TwHIN-BERT tokenizer.
5. Compute **class weights** from the train labels to counter the 4:1 majority/minority imbalance during loss computation.
6. **Hyper-parameter tuning** with Optuna (TPE sampler, median pruner) optimising macro-F1.
7. Retrain the final model with the best hyper-parameters and save the best checkpoint.
8. Evaluate on the held-out validation set (accuracy / F1 / classification report).
9. Apply the model to a new, unseen grievance corpus and compare against manual labels.

## Notes on changes from the previous version
* Added an Optuna study (the previous Optuna code was missing).
* Replaced `val_loss += output.loss` with `.item()` so we don't carry tensors through the autograd graph.
* Added gradient clipping (`max_norm=1.0`) — standard for transformer fine-tuning stability.
* Fixed an inconsistency between the checkpoint *save* path and *load* path.
* Removed an undefined `classified_df` reference; renamed everything to a single `results_df`.
* Added documentation throughout.
* Added a data audit + stratified re-split step (the original split was not properly stratified — class share in val ranged 13%–27%).
* Added class-weighted cross-entropy loss to counter the 4:1 majority/minority imbalance.

# 1. Installations and Imports

In [ ]:
!pip install -q transformers
!pip install -q -U datasets
!pip install -q optuna

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
import json
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt

from torch.optim import AdamW
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
)
from datasets import Dataset, DatasetDict, load_dataset

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from tqdm.notebook import tqdm

# Reduce GPU memory fragmentation on Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Reproducibility
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Configuration

Centralising paths and constants up here makes it easy to tweak the run without hunting through cells.

In [ ]:
# --- Model / experiment naming ---------------------------------------------
MODEL_CHECKPOINT = "Twitter/twhin-bert-base"
MODEL_NAME       = "Twitter"           # short tag used in column names
RUN_NAME         = "20260612_Twitter"  # used in saved file names
NUM_LABELS       = 6
MAX_LENGTH       = 512                 # tokenizer max sequence length

# --- Google Drive paths ----------------------------------------------------
DRIVE_ROOT     = "/content/gdrive/MyDrive/Group 3: palm oil topic classifier"
TRAIN_CSV      = f"{DRIVE_ROOT}/Text Classification Models/Classification Data/Base/BERTopic_TRAIN_80.csv"
VAL_CSV        = f"{DRIVE_ROOT}/Text Classification Models/Classification Data/Base/BERTopic_VAL_20.csv"
MODELS_DIR     = f"{DRIVE_ROOT}/Text Classification Models/Twitter_Saved_Models"
BEST_MODEL_PATH = f"{MODELS_DIR}/{RUN_NAME}_best_model.pt"
OPTUNA_DB_PATH  = f"{MODELS_DIR}/{RUN_NAME}_optuna.db"

# --- New (unseen) classification corpus ------------------------------------
NEW_DATA_PATH   = "/content/gdrive/MyDrive/DSSI/Group 3: palm oil topic classifier/Data/NEW_grievances_formatted.csv"
RESULTS_CSV     = "/content/gdrive/MyDrive/DSSI/Group 3: palm oil topic classifier/Text Classification Models/classified_grievances.csv"

TOPIC_LABELS = [
    "Failed Compensation/Land Rights",   # 0
    "Environmental Impact",              # 1
    "Administrative",                    # 2
    "Deforestation",                     # 3
    "Labour Rights",                     # 4
    "Illegal or Contaminated FFB",       # 5
]

os.makedirs(MODELS_DIR, exist_ok=True)

# 3. Load and Format Data

We load the pre-split 80/20 CSVs and convert them to Hugging Face `Dataset` objects, which integrate cleanly with the tokenizer's `.map()` API.

In [ ]:
train_df      = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VAL_CSV)

print(f"Train: {len(train_df)} rows | Validation: {len(validation_df)} rows")
print("\nTrain label distribution:")
print(train_df['Dominant_Topic'].value_counts().sort_index())
print("\nValidation label distribution:")
print(validation_df['Dominant_Topic'].value_counts().sort_index())

train_ds      = Dataset.from_pandas(train_df)
validation_ds = Dataset.from_pandas(validation_df)

# 3.5 Data Audit & Stratified Re-split

The original split (provided as `BERTopic_TRAIN_80.csv` / `BERTopic_VAL_20.csv`) is **not properly stratified** — per-class share in validation ranges from ~13% (class 5) to ~27% (class 1). Combined with a heavy class imbalance (class 0 has 4× more samples than class 5), this makes per-class metrics noisy and Optuna prone to picking trials that just got lucky on tiny classes.

This section:
1. **Audits** the original split — concatenates train+val, deduplicates by `Text`, checks for leakage (any row appearing in both sets), and reports the per-class distribution.
2. **Re-splits** the combined data with `train_test_split(stratify=...)` so every class has ~20% in validation.
3. Saves the new split CSVs alongside the originals (suffix `_stratified.csv`) so future runs are deterministic.

Set `USE_STRATIFIED_SPLIT = False` to fall back to the original split if you want to reproduce earlier numbers.

In [ ]:
USE_STRATIFIED_SPLIT = True
STRATIFIED_TRAIN_CSV = TRAIN_CSV.replace('.csv', '_stratified.csv')
STRATIFIED_VAL_CSV   = VAL_CSV.replace('.csv',   '_stratified.csv')

# --- Concatenate the original train + val and audit them -----------------
combined_df = pd.concat([train_df, validation_df], ignore_index=True)
print(f'Combined rows (with possible dupes): {len(combined_df)}')

# Leakage check: any Text in BOTH sets?
overlap = set(train_df['Text']).intersection(set(validation_df['Text']))
print(f'Texts appearing in BOTH original train and val: {len(overlap)}')
if overlap:
    print('  (these will be removed by deduplication below)')

# Deduplicate on Text (keep first label seen)
n_before = len(combined_df)
combined_df = combined_df.drop_duplicates(subset=['Text']).reset_index(drop=True)
print(f'After dedup on Text: {len(combined_df)} (removed {n_before - len(combined_df)})')

# Per-class distribution audit (original split vs combined)
audit = pd.DataFrame({
    'orig_train': train_df['Dominant_Topic'].value_counts().sort_index(),
    'orig_val':   validation_df['Dominant_Topic'].value_counts().sort_index(),
    'combined':   combined_df['Dominant_Topic'].value_counts().sort_index(),
}).fillna(0).astype(int)
audit['orig_val_share'] = (audit['orig_val'] / (audit['orig_train'] + audit['orig_val'])).round(3)
print('\nOriginal-split audit:')
print(audit)
print(f'\nSmallest class total: {audit["combined"].min()} — a class this small means single misclassifications swing F1 by tens of points. Treat per-class metrics for this class with caution.')

In [ ]:
if USE_STRATIFIED_SPLIT:
    # Stratified re-split: every class lands at ~20% in val.
    train_df, validation_df = train_test_split(
        combined_df,
        test_size=0.2,
        stratify=combined_df['Dominant_Topic'],
        random_state=SEED,
    )
    train_df      = train_df.reset_index(drop=True)
    validation_df = validation_df.reset_index(drop=True)

    # Persist the new split so this notebook is reproducible alongside other models.
    train_df.to_csv(STRATIFIED_TRAIN_CSV, index=False)
    validation_df.to_csv(STRATIFIED_VAL_CSV, index=False)
    print(f'Stratified split written to:\n  {STRATIFIED_TRAIN_CSV}\n  {STRATIFIED_VAL_CSV}')

print(f'\nUsing split: {"stratified" if USE_STRATIFIED_SPLIT else "original"}')
print(f'Train: {len(train_df)} rows | Validation: {len(validation_df)} rows')

split_audit = pd.DataFrame({
    'train': train_df['Dominant_Topic'].value_counts().sort_index(),
    'val':   validation_df['Dominant_Topic'].value_counts().sort_index(),
}).fillna(0).astype(int)
split_audit['val_share'] = (split_audit['val'] / (split_audit['train'] + split_audit['val'])).round(3)
print('\nPer-class distribution (post re-split):')
print(split_audit)

# Confirm no leakage in the new split
leak = set(train_df['Text']).intersection(set(validation_df['Text']))
assert not leak, f'LEAKAGE: {len(leak)} texts appear in both train and val'
print('\nNo train/val text overlap. Good.')

# Rebuild the HF Datasets from the (possibly new) split.
train_ds      = Dataset.from_pandas(train_df)
validation_ds = Dataset.from_pandas(validation_df)

## Class weights

Even with stratification, class 5 has ~4× fewer samples than class 0. We compute **balanced inverse-frequency weights** from the *training* split (never from the val/test split — that would leak) and pass them into `nn.functional.cross_entropy(..., weight=class_weights)` inside the training loop.

This nudges the model to spend more of its loss budget on the rare classes, which usually helps macro-F1 even though it can slightly hurt overall accuracy.

In [ ]:
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_LABELS),
    y=train_df['Dominant_Topic'].values,
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float, device=device)
print('Class weights (balanced, from training split):')
for i, (name, w) in enumerate(zip(TOPIC_LABELS, class_weights_np)):
    print(f'  {i} ({name}): {w:.3f}')

# 4. Tokenization

We use TwHIN-BERT's tokenizer with `max_length=512` and pad to max length so the dataset can be returned as fixed-shape tensors. For a dataset this small, padding to the max length is cheap and simpler than dynamic padding via a data collator.

Tokenization is wrapped in `prepare_dataset` so the Optuna objective can re-build the dataloaders for different batch sizes without re-tokenizing.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(ds: Dataset, max_length: int = MAX_LENGTH) -> Dataset:
    """Tokenize a HF Dataset's `Text` column and prepare it for the Trainer/DataLoader.

    * Removes the raw text column once tokenized.
    * Renames `Dominant_Topic` -> `labels` (the name expected by HF models).
    * Sets the format to torch tensors.
    """
    tokenized = ds.map(
        lambda ex: tokenizer(
            ex["Text"],
            padding="max_length",
            truncation=True,
            max_length=max_length,
        ),
        batched=False,
    )
    tokenized = tokenized.remove_columns(["Text"])
    tokenized = tokenized.rename_column("Dominant_Topic", "labels")
    tokenized.set_format("torch")
    return tokenized

tokenized_train      = tokenize(train_ds)
tokenized_validation = tokenize(validation_ds)

print("Train features:", tokenized_train.column_names)
print("Sample input_ids shape:", tokenized_train[0]['input_ids'].shape)
print("Sample label:", tokenized_train[0]['labels'].item())

# 5. Training / Evaluation Helpers

Two reusable functions, both used by Optuna **and** the final retraining run:

* `build_model(dropout)` — creates a fresh classifier head on top of TwHIN-BERT with the requested hidden/attention dropout. A fresh model is required for every Optuna trial.
* `train_and_evaluate(...)` — one training run with early stopping. Returns the **best** validation loss, best macro-F1, and the epoch at which the best model was seen. Optionally saves the best checkpoint to disk.

Notable corrections vs. the original loop:
* `val_loss += output.loss.item()` — accumulating Python floats instead of tensors avoids holding the autograd graph in memory.
* `torch.nn.utils.clip_grad_norm_(... 1.0)` — standard gradient clipping for transformer fine-tuning.
* `compute_metrics` returns macro-F1, which is more informative than accuracy for an imbalanced 6-class task.
* Loss is computed manually via `nn.functional.cross_entropy(..., weight=class_weights)` so we can plug in the balanced class weights from §3.5.

In [ ]:
def build_model(dropout: float = 0.1) -> torch.nn.Module:
    """Instantiate a fresh sequence-classification model on the configured device."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=NUM_LABELS,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    )
    return model.to(device)


def _compute_loss(logits: torch.Tensor, labels: torch.Tensor,
                  weights: torch.Tensor | None) -> torch.Tensor:
    """Cross-entropy loss with optional per-class weights."""
    return torch.nn.functional.cross_entropy(logits, labels, weight=weights)


@torch.no_grad()
def evaluate(model: torch.nn.Module, dataloader: DataLoader,
             weights: torch.Tensor | None = None) -> dict:
    """Compute validation loss, accuracy, and macro-F1 over `dataloader`.

    If `weights` is provided the validation loss uses the same class-weighted
    cross-entropy as training, so the two losses are directly comparable.
    """
    model.eval()
    total_loss, n_batches = 0.0, 0
    y_true, y_pred = [], []
    for batch in dataloader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        out = model(input_ids, attention_mask=attention_mask)
        total_loss += _compute_loss(out.logits, labels, weights).item()
        n_batches  += 1
        y_pred.extend(torch.argmax(out.logits, dim=1).cpu().numpy().tolist())
        y_true.extend(labels.cpu().numpy().tolist())
    return {
        "val_loss":   total_loss / max(n_batches, 1),
        "accuracy":   accuracy_score(y_true, y_pred),
        "macro_f1":   f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def train_and_evaluate(
    lr: float,
    weight_decay: float,
    warmup_ratio: float,
    batch_size: int,
    dropout: float,
    num_epochs: int,
    early_stopping_patience: int = 3,
    save_path: str | None = None,
    trial: "optuna.trial.Trial | None" = None,
    verbose: bool = True,
    weights: torch.Tensor | None = None,
) -> dict:
    """Train TwHIN-BERT for sequence classification and return the best metrics.

    Args:
        lr / weight_decay / warmup_ratio / batch_size / dropout / num_epochs:
            Hyper-parameters under tuning.
        early_stopping_patience: stop training if val_loss has not improved
            for this many consecutive epochs.
        save_path: if given, the best checkpoint (lowest val loss) is written here.
        trial: optional Optuna trial used for intermediate reporting + pruning.
        verbose: print per-epoch metrics.
        weights: optional per-class loss weights (1D tensor, length NUM_LABELS).
            When provided, training uses class-weighted cross-entropy and the
            reported validation loss uses the same weighting.

    Returns:
        dict with `best_val_loss`, `best_macro_f1`, `best_epoch`, and the per-epoch
        train/val loss histories (useful for plotting after retraining).
    """
    set_seed(SEED)  # reproducibility within a trial

    model = build_model(dropout=dropout)

    train_loader = DataLoader(tokenized_train,      batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(tokenized_validation, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, eps=1e-8)
    num_training_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(warmup_ratio * num_training_steps),
        num_training_steps=num_training_steps,
    )

    best_val_loss = float("inf")
    best_macro_f1 = -1.0
    best_epoch    = -1
    epochs_since_improvement = 0

    train_losses, val_losses, val_f1s = [], [], []

    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            out  = model(input_ids, attention_mask=attention_mask)
            loss = _compute_loss(out.logits, labels, weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_losses.append(loss.item())

        train_loss = float(np.mean(epoch_losses))
        metrics    = evaluate(model, val_loader, weights=weights)
        train_losses.append(train_loss)
        val_losses.append(metrics["val_loss"])
        val_f1s.append(metrics["macro_f1"])

        if verbose:
            print(
                f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | "
                f"val_loss={metrics['val_loss']:.4f} | "
                f"val_acc={metrics['accuracy']:.3f} | "
                f"val_macro_f1={metrics['macro_f1']:.3f}"
            )

        # Track best model by validation loss (matches original behaviour)
        if metrics["val_loss"] < best_val_loss:
            best_val_loss = metrics["val_loss"]
            best_macro_f1 = metrics["macro_f1"]
            best_epoch    = epoch
            epochs_since_improvement = 0
            if save_path is not None:
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_loss": best_val_loss,
                    "val_macro_f1": best_macro_f1,
                }, save_path)
        else:
            epochs_since_improvement += 1

        # Optuna pruning + intermediate reporting
        if trial is not None:
            trial.report(metrics["macro_f1"], step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if epochs_since_improvement >= early_stopping_patience:
            if verbose:
                print(f"Early stopping at epoch {epoch} (patience={early_stopping_patience}).")
            break

    # Free memory between trials
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "best_val_loss": best_val_loss,
        "best_macro_f1": best_macro_f1,
        "best_epoch":    best_epoch,
        "train_losses":  train_losses,
        "val_losses":    val_losses,
        "val_f1s":       val_f1s,
    }


# 6. Hyperparameter Search

Two paths are provided — **pick one and skip the other**. Both write `best_params`, `MAX_EPOCHS_PER_TRIAL`, and `EARLY_STOPPING_PATIENCE`, which the retraining cell (§7) consumes.

* **§6a — Manual grid search (recommended for this dataset).** 12 runs over learning rate and epoch count, with everything else fixed at the BERT fine-tuning literature defaults. Fast, fully reproducible, easy to explain.
* **§6b — Optuna (advanced).** Bayesian search over 5 hyper-parameters. More powerful, but for 387 labelled rows the headline gain is small and the tuning overfits the validation set more aggressively. Use it if §6a's metric is flat across LRs (i.e., the grid is too coarse to find any signal) or if you want to tune more than just LR + epochs.


## 6a. Manual Grid Search (recommended for small datasets)

Learning rate dominates BERT fine-tuning; epoch count is the next-biggest knob. Everything else stays at sensible defaults from the original BERT paper / HF guidance: `batch_size=8`, `warmup_ratio=0.1`, `weight_decay=0.01`, `dropout=0.1`. Total: **4 LRs × 3 epoch settings = 12 trials**.

If 12 trials is too slow on Colab, trim `GRID['lr']` to `[2e-5, 3e-5, 5e-5]` (the band most favoured in the literature) and/or `GRID['epochs']` to `[3, 5]`.


In [ ]:
# === Option B: small manual grid search over the params that matter most ===
GRID = {
    "lr":     [1e-5, 2e-5, 3e-5, 5e-5],
    "epochs": [3, 5, 8],
}

# Fixed "sensible default" hyperparameters
FIXED = {
    "weight_decay":            0.01,
    "warmup_ratio":            0.10,
    "batch_size":              8,    # use 4 if you hit OOM on Colab
    "dropout":                 0.1,
    "early_stopping_patience": 3,
}

grid_results = []
for lr in GRID["lr"]:
    for epochs in GRID["epochs"]:
        print(f"\n--- lr={lr:g} | epochs={epochs} ---")
        result = train_and_evaluate(
            lr=lr,
            num_epochs=epochs,
            weight_decay=FIXED["weight_decay"],
            warmup_ratio=FIXED["warmup_ratio"],
            batch_size=FIXED["batch_size"],
            dropout=FIXED["dropout"],
            early_stopping_patience=FIXED["early_stopping_patience"],
            save_path=None,           # don't write a checkpoint during search
            trial=None,
            verbose=False,
            weights=class_weights,
        )
        grid_results.append({
            "lr":            lr,
            "epochs":        epochs,
            "best_epoch":    result["best_epoch"],
            "best_val_loss": result["best_val_loss"],
            "best_macro_f1": result["best_macro_f1"],
        })
        print(
            f"   -> best_epoch={result['best_epoch']} "
            f"val_loss={result['best_val_loss']:.4f} "
            f"macro_f1={result['best_macro_f1']:.3f}"
        )

grid_df = pd.DataFrame(grid_results).sort_values("best_macro_f1", ascending=False)
print("\n=== All grid results (sorted by macro-F1) ===")
print(grid_df.to_string(index=False))

# Best config — fed into the final retraining cell.
best_row = grid_df.iloc[0]
best_params = {
    "lr":           float(best_row["lr"]),
    "weight_decay": FIXED["weight_decay"],
    "warmup_ratio": FIXED["warmup_ratio"],
    "batch_size":   FIXED["batch_size"],
    "dropout":      FIXED["dropout"],
}
MAX_EPOCHS_PER_TRIAL    = int(best_row["epochs"])
EARLY_STOPPING_PATIENCE = FIXED["early_stopping_patience"]

print(f"\nBest config: lr={best_params['lr']:g}, epochs={MAX_EPOCHS_PER_TRIAL}, "
      f"macro_f1={best_row['best_macro_f1']:.3f}")


## 6b. Optuna (advanced alternative — skip if you ran §6a)

Bayesian TPE search over 5 hyperparameters. Same `best_params` interface as §6a, so the retraining cell works either way. **Don't run both** — the second one will overwrite `best_params` from the first.

**Search space:**

| Parameter | Range |
|-----------|-------|
| `learning_rate` | log-uniform 1e-6 … 5e-5 |
| `weight_decay`  | uniform   0.0  … 0.3   |
| `warmup_ratio`  | uniform   0.0  … 0.2   |
| `batch_size`    | categorical {2, 4, 8}  |
| `dropout`       | uniform   0.1  … 0.4   |
| `num_epochs`    | fixed at 10 with early stopping (patience 3) |

**Sampler / pruner:** `TPESampler` + `MedianPruner` (stops trials whose intermediate macro-F1 falls below the median of completed trials at the same epoch).

**Persistence:** the study is backed by SQLite on Drive so it can resume across Colab sessions.

In [ ]:
N_TRIALS  = 20    # tune this for your compute budget
MAX_EPOCHS_PER_TRIAL = 10
EARLY_STOPPING_PATIENCE = 3

def objective(trial: optuna.trial.Trial) -> float:
    lr           = trial.suggest_float("lr",            1e-6, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay",  0.0,  0.3)
    warmup_ratio = trial.suggest_float("warmup_ratio",  0.0,  0.2)
    batch_size   = trial.suggest_categorical("batch_size", [2, 4, 8])
    dropout      = trial.suggest_float("dropout",       0.1,  0.4)

    result = train_and_evaluate(
        lr=lr,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        batch_size=batch_size,
        dropout=dropout,
        num_epochs=MAX_EPOCHS_PER_TRIAL,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        save_path=None,         # don't write a checkpoint during tuning
        trial=trial,
        verbose=False,
        weights=class_weights,
    )
    return result["best_macro_f1"]


study = optuna.create_study(
    study_name=RUN_NAME,
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    storage=f"sqlite:///{OPTUNA_DB_PATH}",
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n=== Best trial ===")
print(f"Value (macro-F1): {study.best_value:.4f}")
print("Params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Expose the same variables that §6a sets, so §7 doesn't care which path ran.
best_params = study.best_params

In [ ]:
# Save the best params alongside the study DB for downstream reference
best_params_path = f"{MODELS_DIR}/{RUN_NAME}_best_params.json"
with open(best_params_path, "w") as f:
    json.dump(study.best_params, f, indent=2)
print("Best params written to:", best_params_path)

# Quick visual diagnostics — optional, requires `plotly`
try:
    import plotly.io as pio
    pio.renderers.default = "colab"
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()
except Exception as e:
    print("Skipping Optuna plots:", e)

# 7. Retrain Final Model with Best Hyper-parameters

Whichever path you took above (§6a or §6b) selected its best configuration **on the validation set**, so the validation metric is *biased* — we picked the run that fits it best. To get a final "production" checkpoint we retrain once with those params and save it. Downstream you should evaluate it on **truly held-out** test data (e.g., the new corpus with manual labels in §9a).

In [ ]:
# `best_params`, `MAX_EPOCHS_PER_TRIAL`, and `EARLY_STOPPING_PATIENCE`
# were set by whichever search path you ran (§6a or §6b).
print("Retraining with:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"  num_epochs: {MAX_EPOCHS_PER_TRIAL}")
print(f"  early_stopping_patience: {EARLY_STOPPING_PATIENCE}")

final_result = train_and_evaluate(
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
    warmup_ratio=best_params["warmup_ratio"],
    batch_size=best_params["batch_size"],
    dropout=best_params["dropout"],
    num_epochs=MAX_EPOCHS_PER_TRIAL,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    save_path=BEST_MODEL_PATH,
    trial=None,
    verbose=True,
    weights=class_weights,
)

print("\nBest checkpoint at epoch:", final_result["best_epoch"])
print("Best val loss   :", final_result["best_val_loss"])
print("Best macro F1   :", final_result["best_macro_f1"])
print("Saved to        :", BEST_MODEL_PATH)

In [ ]:
# Plot training curves
epochs = list(range(len(final_result["train_losses"])))
plt.style.use("fivethirtyeight")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, final_result["train_losses"], label="Train loss")
ax[0].plot(epochs, final_result["val_losses"],   label="Val loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend(); ax[0].set_title("Loss")
ax[1].plot(epochs, final_result["val_f1s"], label="Val macro-F1", color="tab:green")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Macro F1"); ax[1].legend(); ax[1].set_title("Validation macro-F1")
plt.tight_layout(); plt.show()

# 8. Load the Best Checkpoint and Evaluate on the Validation Set

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']} "
      f"(val_loss={checkpoint['val_loss']:.4f}, val_macro_f1={checkpoint.get('val_macro_f1', float('nan')):.4f})")

In [ ]:
# Run inference on the validation set in a single big batch (n=78)
val_loader = DataLoader(tokenized_validation, batch_size=len(tokenized_validation))

all_logits, y_true = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].cpu().numpy()
        out = model(input_ids, attention_mask=attention_mask)
        all_logits.append(out.logits)
        y_true.extend(labels.tolist())

val_logits = torch.cat(all_logits, dim=0)
y_pred     = torch.argmax(val_logits, dim=1).cpu().numpy()
assert len(y_pred) == len(y_true)

print("Accuracy        :", accuracy_score(y_true, y_pred))
print("F1 (macro)      :", f1_score(y_true, y_pred, average="macro"))
print("F1 (weighted)   :", f1_score(y_true, y_pred, average="weighted"))
print("F1 (micro)      :", f1_score(y_true, y_pred, average="micro"))
print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=[f"{name} ({i})" for i, name in enumerate(TOPIC_LABELS)],
    zero_division=0,
))

# 9. Predict on a New Unseen Corpus

We now apply the best checkpoint to a separate corpus of grievances (`NEW_grievances_formatted.csv`) and append the predictions as a new column on `classified_grievances.csv`. If the manual gold labels are available in that file we also report the comparison metrics.

In [ ]:
new_df = pd.read_csv(NEW_DATA_PATH)
print(f"Loaded {len(new_df)} entries for classification.")
print("Columns:", new_df.columns.tolist())

TEXT_COLUMN = "Text"
new_df[TEXT_COLUMN] = new_df[TEXT_COLUMN].fillna("").astype(str)

new_ds = Dataset.from_pandas(new_df[[TEXT_COLUMN]])
tokenized_new = new_ds.map(
    lambda ex: tokenizer(
        ex[TEXT_COLUMN],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    ),
)
tokenized_new = tokenized_new.remove_columns([TEXT_COLUMN])
tokenized_new.set_format("torch")

INFERENCE_BATCH_SIZE = max(best_params.get("batch_size", 4), 4)
inference_loader = DataLoader(tokenized_new, batch_size=INFERENCE_BATCH_SIZE)

model.eval()
all_predictions = []
with torch.no_grad():
    for batch in tqdm(inference_loader, desc="Classifying"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(out.logits, dim=1).cpu().numpy()
        all_predictions.extend(preds.tolist())

print(f"Generated {len(all_predictions)} predictions.")

In [ ]:
# Merge predictions into the results CSV
label_map = dict(enumerate(TOPIC_LABELS))

results_df = pd.read_csv(RESULTS_CSV)
results_df[f"{RUN_NAME}_label"] = all_predictions
results_df[f"{RUN_NAME}_topic"] = results_df[f"{RUN_NAME}_label"].map(label_map)

print("Label distribution:")
print(results_df[f"{RUN_NAME}_label"].value_counts().sort_index())

results_df.to_csv(RESULTS_CSV, index=False)
print(f"\nWrote: {RESULTS_CSV}")
results_df[["Text", f"{RUN_NAME}_label", f"{RUN_NAME}_topic"]].head(10)

In [ ]:
# Bar chart of topic counts on the new corpus
plt.figure(figsize=(10, 6))
results_df[f"{RUN_NAME}_topic"].value_counts().plot(kind="bar", edgecolor="black")
plt.title("Number of Grievances per Topic (new corpus)")
plt.xlabel("Topic"); plt.ylabel("Count")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 9a. Evaluation on the New Corpus (if manual labels exist)

In [ ]:
if "manual_label" in results_df.columns:
    true_labels = results_df["manual_label"].astype(int).values
    pred_labels = np.array(all_predictions)
    print("Evaluation on new corpus:")
    print("  F1 (macro)    :", f1_score(true_labels, pred_labels, average="macro"))
    print("  F1 (weighted) :", f1_score(true_labels, pred_labels, average="weighted"))
    print("  F1 (micro)    :", f1_score(true_labels, pred_labels, average="micro"))
    print("  Accuracy      :", accuracy_score(true_labels, pred_labels))
    print("\nClassification report:")
    print(classification_report(true_labels, pred_labels, target_names=TOPIC_LABELS, zero_division=0))
else:
    print("No `manual_label` column found in results CSV — skipping new-corpus evaluation.")

# 10. Cross-model Visualisation

If predictions from other architectures (DistilBERT, ELECTRA, SpanBERT, …) already live in `results_df`, plot the label distributions side by side against the manual gold labels. Missing columns are simply skipped.

In [ ]:
candidate_cols = {
    "Manual":     "manual_label",
    "DistilBERT": "DistilBERT_label",
    "Electra":    "Electra_label",
    "Twitter":    f"{RUN_NAME}_label",
    "SpanBERT":   "SpanBERT_label",
}
present = {k: v for k, v in candidate_cols.items() if v in results_df.columns}

if present:
    combined = pd.DataFrame({
        name: results_df[col].value_counts() for name, col in present.items()
    }).fillna(0).sort_index()
    combined.plot(kind="bar", figsize=(12, 6), edgecolor="black")
    plt.title("Number of Grievances per Topic: Manual vs Models")
    plt.xlabel("Topic"); plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Label Source"); plt.tight_layout(); plt.show()
else:
    print("No matching label columns in results_df.")